# BIW Notebook
支持时间细胞风格 Sliding Window + Stride。


In [13]:
import os, re
from glob import glob
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

device='cuda' if torch.cuda.is_available() else 'cpu'
print('device=', device)


# 2. 将其加入 Python 路径
import sys
import os
sys.path.append(os.path.join(os.getcwd(), 'torch_tem'))

# 3. 导入 TEM 模型
# 注意：具体导入路径取决于 repo 结构，通常是 model.py
from torch_tem.model import Model as TEM_Model

# 2. 从 parameters.py 导入 Parameters 类
from torch_tem.parameters import parameters

# 3. 实例化并覆盖参数
# TEM 的默认参数通常是为 Grid World 设计的，我们需要修改以适配 ResNet 和 GPU
tem_params = parameters()

# --- 关键修改：适配你的数据 ---
tem_params.n_x = 256            # 对应 ResNet 的输出维度 (embed_dim)
tem_params.n_actions = 1        # 我们只有一个虚拟动作 "Forward"
tem_params.device = device      # 使用你的 'cuda' 或 'cpu'

# --- 其它可选修改 (根据显存大小调整) ---
# 如果显存不够，可以调小这些值
# tem_params.n_p = 50 
# tem_params.n_g = 50 

print("TEM Parameters loaded from file and updated.")

device= cuda


AttributeError: 'dict' object has no attribute 'n_x'

## 1. Multi-Trajectory Dataset (Time-Cell Style)
遍历每个轨迹子文件夹，生成 sequence 样本。


In [ ]:
def sort_key(path):
    nums = re.findall(r'\d+', os.path.basename(path))
    return int(nums[-1]) if nums else 0

# 修改第 1 部分的 Dataset 代码
class GoStanfordMultiTrajDataset(Dataset):
    def __init__(self, root, split='train', seq_len=8, stride=2, transform=None, max_traj=None, max_frames=None):
        self.seq_len = seq_len
        self.stride = stride
        self.transform = transform

        split_dir = os.path.join(root, split) if split else root
        traj_dirs = sorted([d for d in glob(split_dir + '/*') if os.path.isdir(d)])
        
        if max_traj:
            traj_dirs = traj_dirs[:max_traj]

        self.sequences = []
        # 新增：保存原始轨迹的完整路径信息，用于评估和画图
        self.raw_trajectories = [] 

        for traj in traj_dirs:
            # 获取该轨迹下所有图片并排序
            frames = sorted(glob(os.path.join(traj, '*.jpg')), key=sort_key)
            if max_frames:
                frames = frames[:max_frames]
            
            # 只有长度足够的轨迹才被记录
            if len(frames) >= seq_len:
                # 保存原始轨迹信息
                self.raw_trajectories.append({
                    'id': os.path.basename(traj),
                    'paths': frames
                })
                
                # 生成训练用的滑动窗口
                for i in range(0, len(frames)-seq_len, stride):
                    self.sequences.append(frames[i:i+seq_len])

        print(f"Loaded {len(self.sequences)} sequences from {len(self.raw_trajectories)} trajectories")

    def __len__(self): return len(self.sequences)

    def __getitem__(self, idx):
        imgs = []
        for path in self.sequences[idx]:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            imgs.append(img)
        return torch.stack(imgs)


## 2. Backbone & World Model


In [ ]:
class CognitiveBackbone(nn.Module):
    def __init__(self, arch='resnet18', pretrained=True, embed_dim=256):
        super().__init__()
        if arch=='resnet18': net=models.resnet18(pretrained=pretrained); dim=512
        elif arch=='resnet50': net=models.resnet50(pretrained=pretrained); dim=2048
        else: raise ValueError
        self.encoder=nn.Sequential(*list(net.children())[:-2])
        self.pool=nn.AdaptiveAvgPool2d((1,1))
        self.projector=nn.Sequential(nn.Linear(dim,512), nn.ReLU(), nn.Linear(512,embed_dim))
    def forward(self,x):
        f=self.encoder(x); f=self.pool(f).flatten(1)
        z=F.normalize(self.projector(f), dim=1)
        return z

class WorldModel(nn.Module):
    def __init__(self, d=256):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(d,512), nn.ReLU(), nn.Linear(512,d))
    def forward(self,z): return self.net(z)


## 3、创建 TEM 适配器 (TEMWrapper)

In [ ]:
# TEM 配置参数 (简化版，适配 continuous data 需要调整)
tem_config = {
    'n_x': 256,              # Sensory input dimension (匹配 ResNet 输出)
    'n_actions': 1,          # 我们只构造了 1 个虚拟动作 (Forward)
    'n_locations': 50,       # 预估环境中的状态/位置数量 (海马体容量)
    'n_g': 100,              # Grid cell 模块的数量
    'n_p': 100,              # Path integrator 维度
    'n_f': 100,              # Feature 维度
    'g_frequencies': 5,      # Grid cell 频率数
    'hebbian_learning_rate': 0.1, # 记忆更新率
    'device': device
}

class TEMWrapper(nn.Module):
    def __init__(self, input_dim, params):
        super().__init__()
        self.params = params
        
        # 直接将 params 对象传给 TEM_Model
        # 原始 repo 的 Model 类就是设计为接受 Parameters 对象的
        self.tem = TEM_Model(self.params)
        
        # 投影层：如果 ResNet 输出维度与 TEM 设置的 n_x 不同，则需要这一层
        # 这里我们已经在上面设置了 tem_params.n_x = input_dim，所以理论上只是一个 Identity
        if input_dim != params.n_x:
            self.input_proj = nn.Linear(input_dim, params.n_x)
        else:
            self.input_proj = nn.Identity()
            
        self.n_actions = params.n_actions

    def forward(self, z_seq):
        """
        z_seq: [Batch, Time, Dim]
        """
        batch_size, seq_len, dim = z_seq.shape
        
        # 1. 特征处理
        # ResNet: [B, T, D] -> TEM need: [T, B, n_x]
        x = self.input_proj(z_seq).permute(1, 0, 2) 
        
        # 2. 构造虚拟动作 (Dummy Actions)
        # 构造全 0 的动作向量，仅激活第 0 个维度 (Forward)
        actions = torch.zeros(seq_len, batch_size, self.n_actions).to(z_seq.device)
        actions[:, :, 0] = 1.0 
        
        # 3. 输入 TEM
        outputs = self.tem(x, actions)
        
        return outputs

## 3. Tiny TEM Memory (Placeholder)

In [ ]:
class TinyTEM(nn.Module):
    def __init__(self, d=256, capacity=5000): super().__init__(); self.capacity=capacity; self.reset()
    def reset(self): self.mem=[]
    def forward(self,z):
        self.mem.append(z.detach().cpu())
        if len(self.mem)>self.capacity: self.mem=self.mem[-self.capacity:]
        return z
    def get_map(self): return torch.cat(self.mem, dim=0) if len(self.mem)>0 else None


## 4. Training


In [ ]:
ROOT = os.environ.get('GOSTANFORD_ROOT', '/media/zhen/Data/Datasets/nomad_data/go_stanford')
SEQ=20; STRIDE=2; BATCH=8; EPOCH=20

tfm=T.Compose([
    T.Resize((224,224)), T.ToTensor(),
    T.Normalize([.485,.456,.406],[.229,.224,.225])
])

ds=GoStanfordMultiTrajDataset(ROOT, split='', seq_len=SEQ, stride=STRIDE, transform=tfm,
                              max_traj=10, max_frames=150)
dl=DataLoader(ds, batch_size=BATCH, shuffle=True)

# enc=CognitiveBackbone(embed_dim=256).to(device)
# wm=WorldModel(256).to(device)
# tem=TinyTEM(256)

# opt=torch.optim.Adam(list(enc.parameters())+list(wm.parameters()), lr=1e-4)
# mse=nn.MSELoss()

# for ep in range(EPOCH):
#     tot=0
#     for seq in dl:
#         B,T,C,H,W=seq.shape
#         seq=seq.to(device)
#         z=enc(seq.view(B*T,C,H,W)).view(B,T,-1)
#         pred=wm(z[:,:-1])
#         loss=mse(pred, z[:,1:])
#         opt.zero_grad(); loss.backward(); opt.step()
#         tot+=loss.item()
#     print('epoch',ep,'loss',tot/len(dl))

# --- 初始化模型 ---
# 1. 视觉编码器 (保持不变)
enc = CognitiveBackbone(embed_dim=256).to(device)

# 2. TEM 模型 (传入我们在第1步配置好的 tem_params 对象)
tem_wrapper = TEMWrapper(input_dim=256, params=tem_params).to(device)

# 3. 优化器 (包含两者的参数)
opt = torch.optim.Adam(list(enc.parameters()) + list(tem_wrapper.parameters()), lr=1e-4)

# --- 训练循环 ---
for ep in range(EPOCH):
    total_loss = 0
    for seq in dl:
        # seq shape: [Batch, Time, Channel, H, W]
        B, T, C, H, W = seq.shape
        seq = seq.to(device)
        
        # 1. 通过 ResNet 提取视觉特征
        # [B, T, C, H, W] -> [B*T, C, H, W] -> [B*T, 256] -> [B, T, 256]
        z = enc(seq.view(B*T, C, H, W)).view(B, T, -1)
        
        # 2. 输入 TEM
        # 注意：TEM 的输出通常包含对下一个时刻的预测 (gen_x)
        tem_output = tem_wrapper(z)
        
        # 3. 计算 Loss
        # TEM 通常输出的是重构的 x (x_gen)
        # 假设 tem_output[0] 是预测的 x 序列
        x_pred = tem_output[0].permute(1, 0, 2) # 变回 [B, T, Dim]
        
        # 我们的目标是预测下一个时刻，所以对比 x_pred[:, :-1] 和 z[:, 1:]
        # 或者 TEM 内部是自回归的，x_pred 已经是预测值
        # 这里使用简单的 MSE Loss 来约束特征预测
        loss = F.mse_loss(x_pred, z) 
        
        # *进阶*：TEM 原库通常有自己的 ELBO Loss (tem_output 包含 loss 项)
        # 如果 model 返回了 loss，直接使用： loss = tem_output['loss']
        
        opt.zero_grad()
        loss.backward()
        opt.step()
        
        total_loss += loss.item()
        
    print(f'Epoch {ep}, Loss: {total_loss/len(dl)}')


## 5. Cognitive Map Visualization (One Trajectory)


In [ ]:
tem.reset()
sample = ds[0].unsqueeze(0).to(device)
B,T,C,H,W=sample.shape
with torch.no_grad():
    z=enc(sample.view(B*T,C,H,W)).view(B,T,-1)[0]
    for t in range(T): tem(z[t:t+1])

MAP=tem.get_map().numpy()
Z2=PCA(2).fit_transform(MAP)

plt.figure(figsize=(5,4))
plt.plot(Z2[:,0], Z2[:,1], '-o')
plt.title('Cognitive Map (Multi-Traj)')
plt.axis('equal'); plt.show()


In [ ]:
# 在文件末尾添加以下新功能

# 替换第 5 部分的 MultiTrajectoryEvaluator
from torch.utils.data import TensorDataset
from sklearn.decomposition import PCA
from torch.utils.data import TensorDataset, DataLoader
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image

class MultiTrajectoryEvaluator:
    def __init__(self, encoder, world_model, tem_capacity=5000, eval_batch_size=32):
        self.encoder = encoder
        self.world_model = world_model
        self.tem = TinyTEM(256, capacity=tem_capacity)
        self.eval_batch_size = eval_batch_size
        
    def evaluate_trajectory(self, trajectory_data, device):
        """
        分批次评估单个轨迹
        """
        self.encoder.eval()
        self.world_model.eval()
        
        T_total = trajectory_data.size(0)
        if T_total < 2: return None
        
        # --- 分批次通过 Encoder ---
        temp_dataset = TensorDataset(trajectory_data)
        temp_loader = DataLoader(temp_dataset, batch_size=self.eval_batch_size, shuffle=False)
        
        z_list = []
        
        with torch.no_grad():
            for (batch_imgs,) in temp_loader:
                batch_imgs = batch_imgs.to(device)
                z_batch = self.encoder(batch_imgs) 
                z_list.append(z_batch)
            
            z = torch.cat(z_list, dim=0) 
            
            # 预测下一时刻
            if T_total > 1:
                z_dev = z.to(device)
                z_pred = self.world_model(z_dev[:-1])
                z_true = z_dev[1:]
                
                mse_error = F.mse_loss(z_pred, z_true).item()
                cos_sim = F.cosine_similarity(z_pred, z_true, dim=1).mean().item()
            else:
                mse_error = 0; cos_sim = 1

            # --- 更新 TEM 记忆 (确保在 CPU 上进行以节省显存) ---
            for t in range(T_total):
                self.tem(z[t:t+1].detach().cpu()) 
                
        return {
            'length': T_total,
            'mse_error': mse_error,
            'cosine_similarity': cos_sim
        }
    
    def build_cognitive_map(self, dataset, device, max_trajectories=None):
        self.tem.reset()
        trajectory_metrics = []
        processed_count = 0
        
        # 使用 dataset.raw_trajectories
        traj_list = dataset.raw_trajectories
        if max_trajectories:
            traj_list = traj_list[:max_trajectories]
            
        print(f"Processing {len(traj_list)} full trajectories...")
        
        for traj_info in traj_list:
            traj_id = traj_info['id']
            paths = traj_info['paths']
            
            frames = []
            for path in paths:
                try:
                    img = Image.open(path).convert('RGB')
                    if dataset.transform:
                        img = dataset.transform(img)
                    frames.append(img)
                except Exception as e:
                    print(f"Error loading {path}: {e}")
            
            if not frames: continue
                
            trajectory_tensor = torch.stack(frames)
            metrics = self.evaluate_trajectory(trajectory_tensor, device)
            
            if metrics:
                metrics['trajectory_id'] = traj_id
                trajectory_metrics.append(metrics)
                processed_count += 1
                print(f"  Traj {traj_id}: Length={metrics['length']}, MSE={metrics['mse_error']:.4f}")
        
        # --- 补回缺失的 PCA 计算部分 ---
        cognitive_map = self.tem.get_map()
        map_2d = None
        map_3d = None
        
        if cognitive_map is not None and len(cognitive_map) > 0:
            # 转换为 numpy
            map_numpy = cognitive_map.detach().cpu().numpy()
            
            # 计算 2D 投影
            if map_numpy.shape[0] >= 2:
                try:
                    pca_2d = PCA(n_components=2)
                    map_2d = pca_2d.fit_transform(map_numpy)
                except Exception as e:
                    print(f"PCA 2D failed: {e}")
            
            # 计算 3D 投影 (可选)
            if map_numpy.shape[0] >= 3:
                try:
                    pca_3d = PCA(n_components=3)
                    map_3d = pca_3d.fit_transform(map_numpy)
                except Exception:
                    pass

        return {
            'cognitive_map': cognitive_map,
            'trajectory_metrics': trajectory_metrics,
            'processed_trajectories': processed_count,
            'map_2d': map_2d,  # 现在这里有值了
            'map_3d': map_3d
        }

def visualize_multi_trajectory_map(evaluation_result, title="Multi-Trajectory Cognitive Map"):
    """
    可视化多轨迹认知地图
    
    Args:
        evaluation_result: 评估结果字典
        title: 图标题
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # 2D认知地图
    if evaluation_result['map_2d'] is not None:
        axes[0].scatter(evaluation_result['map_2d'][:, 0], 
                       evaluation_result['map_2d'][:, 1], 
                       alpha=0.6, s=20)
        axes[0].set_title(f"{title} (2D)")
        axes[0].set_xlabel("PC1")
        axes[0].set_ylabel("PC2")
        axes[0].grid(True)
    
    # 性能指标统计
    if evaluation_result['trajectory_metrics']:
        metrics = evaluation_result['trajectory_metrics']
        mse_errors = [m['mse_error'] for m in metrics]
        cos_sims = [m['cosine_similarity'] for m in metrics]
        
        axes[1].hist(mse_errors, bins=20, alpha=0.7, label='MSE Error')
        axes[1].set_title("Prediction Error Distribution")
        axes[1].set_xlabel("MSE")
        axes[1].set_ylabel("Frequency")
        axes[1].legend()
        axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

def print_evaluation_summary(evaluation_result):
    """
    打印评估摘要
    
    Args:
        evaluation_result: 评估结果字典
    """
    metrics = evaluation_result['trajectory_metrics']
    if not metrics:
        print("No trajectories were processed.")
        return
    
    avg_mse = np.mean([m['mse_error'] for m in metrics])
    avg_cos_sim = np.mean([m['cosine_similarity'] for m in metrics])
    total_states = sum([m['length'] for m in metrics])
    
    print("=" * 50)
    print("MULTI-TRAJECTORY EVALUATION SUMMARY")
    print("=" * 50)
    print(f"Processed trajectories: {evaluation_result['processed_trajectories']}")
    print(f"Total states in map: {total_states}")
    print(f"Average prediction MSE: {avg_mse:.6f}")
    print(f"Average cosine similarity: {avg_cos_sim:.4f}")
    print("=" * 50)


In [ ]:

# 重新初始化 Dataset (为了让新的 __init__ 生效)
ds = GoStanfordMultiTrajDataset(ROOT, split='', seq_len=SEQ, stride=STRIDE, transform=tfm,
                              max_traj=10, max_frames=150)

# 创建评估器 (设置 eval_batch_size 以控制显存)
evaluator = MultiTrajectoryEvaluator(enc, wm, tem_capacity=10000, eval_batch_size=32)

# 执行
try:
    eval_result = evaluator.build_cognitive_map(ds, device, max_trajectories=5)
    print_evaluation_summary(eval_result)
    visualize_multi_trajectory_map(eval_result)
except Exception as e:
    print(f"An error occurred: {e}")
    import traceback
    traceback.print_exc()

    
# 也可以单独查看某个轨迹的表现
if eval_result['trajectory_metrics']:
    first_traj = eval_result['trajectory_metrics'][0]
    print(f"\nFirst trajectory details:")
    print(f"  ID: {first_traj['trajectory_id']}")
    print(f"  Length: {first_traj['length']} states")
    print(f"  MSE error: {first_traj['mse_error']:.6f}")
    print(f"  Cosine similarity: {first_traj['cosine_similarity']:.4f}")
